# (Feature Engineering + Baseline Model)

import libs and data

In [74]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../data/processed/clean_clothing_sales.csv")
df.sample(5)


,product_id,product_position,promotion,product_category,seasonal,sales_volume,brand,url,name,description,price,currency,terms,section,season,material,origin
12907,212287,Front of Store,0,clothing,0,802,Zara,https://www.zara.com/us/en/faux-suede-bomber-j...,STRUCTURED BAGGY FIT JEANS LIMITED EDITION ECRU,Jacket made of dense technical fabric. Lapel c...,25.99,USD,jackets,MAN,Spring,Cotton,Turkey
16466,215846,Aisle,0,clothing,0,539,Zara,https://www.zara.com/us/en/flared-fit-cargo-je...,PLEATED TEXTURED WEAVE OVERSHIRT CHARCOAL,Running shoes. Upper in a combination of piece...,119.99,USD,sweaters,MAN,Autumn,Cotton,Brazil
13643,213023,Aisle,0,clothing,0,751,Zara,https://www.zara.com/us/en/plaid-overshirt-p08...,RIBBED DOUBLE FACED JACKET BURGUNDY,Relaxed fit overshirt. Lapel collar and long s...,49.00,USD,jackets,MAN,Spring,Linen Blend,Spain
7388,206768,End-cap,1,clothing,1,1721,Zara,https://www.zara.com/us/en/heart-print-t-shirt...,UTILITY KNIT OPEN BACK PEARLY SWEATER BURGUNDY,Full cut T-shirt with round neck and short sle...,54.99,USD,t-shirts,WOMAN,Summer,Linen,India
9359,208739,Aisle,0,clothing,0,874,Zara,https://www.zara.com/us/en/multi-pieced-retro-...,BELTED VINTAGE EFFECT LEATHER BOMBER JACKET BROWN,Slim fit jacket made of viscose blend fabric. ...,22.95,USD,jackets,WOMAN,Autumn,Cotton,Turkey


**Drop Useless Columns**

The columns "brand", "seasonal", "currency" and "product_category" has only one unique value throughout all records and hence dropped because they have no predictive power. 
"url" and "product_id" is an identifier.

In [75]:
cols_to_drop = [
    "brand",
    "product_category",
    "currency",
    "url",
    "product_id"
]

df = df.drop(columns=cols_to_drop)
df.head()


,product_position,promotion,seasonal,sales_volume,name,description,price,terms,section,season,material,origin
0,Aisle,1,1,1243,BASIC PUFFER JACKET,Puffer jacket made of tear-resistant ripstop f...,78.99,jackets,MAN,Winter,Polyester,Brazil
1,Aisle,1,0,1429,TUXEDO JACKET,Straight fit blazer. Pointed lapel collar and ...,14.99,jackets,MAN,Autumn,Cotton,Turkey
2,End-cap,1,1,1168,SLIM FIT SUIT JACKET,Slim fit jacket. Notched lapel collar. Long sl...,71.95,jackets,WOMAN,Autumn,Polyester,Morocco
3,Aisle,1,0,1348,STRETCH SUIT JACKET,Slim fit jacket made of viscose blend fabric. ...,30.99,jackets,MAN,Spring,Polyester,China
4,End-cap,1,1,1602,DOUBLE FACED JACKET,Jacket made of faux leather faux shearling wit...,22.99,jackets,WOMAN,Winter,Wool Blend,China


# Text Analysis and Feature Creation

In [76]:
# Create basic text features (part of clean data)
df['desc_len'] = df['description'].str.len().fillna(0)
df['name_len'] = df['name'].str.len().fillna(0)
df['name_word_count'] = df['name'].str.split().apply(lambda x: len(x) if isinstance(x, list) else 0)

df['desc_word_count'] = df['description'].str.split().apply(lambda x: len(x) if isinstance(x, list) else 0)
df['price_per_word'] = df['price'] / (df['desc_word_count'] + 1) # +1 to avoid division by zero
df['price_per_name_word'] = df['price'] / (df['name_word_count'] + 1)

# Target transformation (for potential modeling, but saved with raw data)
df['sales_volume_log'] = np.log1p(df['sales_volume'])

# Text analysis - show example description lengths
df[['desc_len', 'desc_word_count']].describe()

,desc_len,desc_word_count
count,20252.000000,20252.000000
mean,132.655145,20.989976
std,51.321476,8.047790
min,0.000000,0.000000
25%,92.000000,15.000000
50%,132.000000,21.000000
75%,168.000000,27.000000
max,313.000000,48.000000


In [77]:
df.columns

Index(['product_position', 'promotion', 'seasonal', 'sales_volume', 'name',
       'description', 'price', 'terms', 'section', 'season', 'material',
       'origin', 'desc_len', 'name_len', 'name_word_count', 'desc_word_count',
       'price_per_word', 'price_per_name_word', 'sales_volume_log'],
      dtype='object')

In [78]:
# Define features and target (using new text features created in Notebook 01)
TARGET = 'sales_volume_log'
TEXT_FEATURES = ['description', 'name']
NUMERIC_FEATURES = [
    'price', 
    'desc_len', 
    'name_len', 
    'desc_word_count',
    'price_per_word',
    'name_word_count',
    'price_per_name_word'
    
]
CATEGORICAL_FEATURES = [
    'product_position',
    'section',
    'season',
    'material',
    'origin',
    'terms'
]

BINARY_FEATURES = [ 
    'promotion',
    'seasonal' 
]

for col in CATEGORICAL_FEATURES:
    df[col] = df[col].astype(str)


# Train-Test Split 
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42)

# Define X/y matrices
all_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES + BINARY_FEATURES + TEXT_FEATURES
X_train = train_df[all_features]
y_train = train_df[TARGET].values 
X_test = test_df[all_features]
y_test_log = test_df[TARGET].values # Save y_test as log-transformed

Feature Engineering: column transformer

In [79]:
# Text pipeline for description (TF-IDF)
tfidf = TfidfVectorizer(max_features=4000, ngram_range=(1,2), stop_words='english')
# Text vectorizer for name (TF-IDF)
tfidf_name = TfidfVectorizer(max_features=2000, ngram_range=(1,2), stop_words='english')

# Categorical pipeline: OneHotEncoding
cat_ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Ensure all data in the column is converted to string
def to_string(X):
    return X.astype(str)

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUMERIC_FEATURES),
    
    ('bin', 'passthrough', BINARY_FEATURES), 
    
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
   
    ('desc_txt', Pipeline([
        ('str_convert', FunctionTransformer(to_string, validate=False)),
        ('tfidf', TfidfVectorizer(max_features=4000, ngram_range=(1,2), stop_words='english'))
    ]), 'description'),
    
    ('name_txt', Pipeline([
        ('str_convert', FunctionTransformer(to_string, validate=False)),
        ('tfidf', TfidfVectorizer(max_features=2000, ngram_range=(1,2), stop_words='english'))
    ]), 'name')
], remainder='drop', verbose=False)

print("Preprocessor ready with StandardScaler, OneHotEncoder, and TfidfVectorizer.")

Preprocessor ready with StandardScaler, OneHotEncoder, and TfidfVectorizer.


In [80]:
print(df.columns)  # check if 'name' exists


Index(['product_position', 'promotion', 'seasonal', 'sales_volume', 'name',
       'description', 'price', 'terms', 'section', 'season', 'material',
       'origin', 'desc_len', 'name_len', 'name_word_count', 'desc_word_count',
       'price_per_word', 'price_per_name_word', 'sales_volume_log'],
      dtype='object')


RandomForest Baseline and Metrics

In [81]:
# Define the RandomForest baseline model
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# Build the full pipeline
baseline_pipeline = Pipeline(steps=[
    ('pre', preprocessor),
    ('model', rf)
])

# Fit on training data
print("Fitting RandomForest Baseline...")
baseline_pipeline.fit(X_train, y_train)

# Predict and evaluate on test set
y_pred_log = baseline_pipeline.predict(X_test)

# 1. Inverse transform the predictions back to the original scale
y_pred = np.expm1(y_pred_log)

# 2. Use the original sales_volume values from the test set for comparison
y_test_original = test_df['sales_volume'].values 

# 3. Calculate metrics using the original scale values
mae = mean_absolute_error(y_test_original, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred))
r2 = r2_score(y_test_original, y_pred)


print("\n--- RandomForest Baseline Test Metrics ---")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")

Fitting RandomForest Baseline...

--- RandomForest Baseline Test Metrics ---
MAE: 61.2399
RMSE: 78.0582
R2: 0.9307


Save the baseline model

In [82]:
# Save the trained baseline pipeline (preprocessor + RF model)
os.makedirs("../models", exist_ok=True)
joblib.dump(baseline_pipeline, "../models/sales_rf_baseline_pipeline.pkl")
print("Saved RandomForest baseline pipeline to ../models/sales_rf_baseline_pipeline.pkl")

Saved RandomForest baseline pipeline to ../models/sales_rf_baseline_pipeline.pkl
